# Mejorando YOLO con SAHI — Benchmark sobre VisDrone
### FLISOL 2026 · Detección de objetos pequeños en vista aérea urbana

Comparamos el **mismo modelo** con y sin SAHI sobre VisDrone (drones, cientos de objetos diminutos por imagen) y medimos **mAP**, con foco en **AP de objetos pequeños**.

**Antes de empezar:** `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)`.

## 1. Setup

In [ ]:
!pip install -q ultralytics sahi pycocotools supervision

# Clona el repo del proyecto (reemplaza por tu URL)
!git clone https://github.com/TU-USUARIO/yolo-sahi-flisol2026.git || echo 'ya clonado'
%cd yolo-sahi-flisol2026

import torch
print('GPU:', torch.cuda.is_available())

## 2. Descargar VisDrone val + convertir a COCO

In [ ]:
!python scripts/download_visdrone.py

## 3. El modelo

Para el **mAP real** necesitas un YOLO entrenado en VisDrone (`src/train_visdrone.py`, correr ANTES de la charla).
Sube tu `best.pt` a `weights/visdrone_yolo11s.pt`.

Si solo quieres la demo VISUAL rápida, deja que use `yolo11s.pt` (COCO) — los recuadros igual lucen el efecto del slicing.

In [ ]:
import os
MODEL = 'weights/visdrone_yolo11s.pt' if os.path.exists('weights/visdrone_yolo11s.pt') else 'yolo11s.pt'
print('Usando modelo:', MODEL)
print('(mAP real requiere el modelo VisDrone; con yolo11s.pt solo vale la galería visual)')

## 4. Galería antes/después (impacto visual)

Genera comparativas lado a lado y las ordena por cuántos objetos extra encontró SAHI.

In [ ]:
!python src/gallery.py --model $MODEL --limit 15 --top 5

from PIL import Image
import glob
best = sorted(glob.glob('outputs/gallery/*.jpg'))[0]
print(best)
Image.open(best)

## 5. Los números: mAP YOLO vs YOLO + SAHI

Evalúa ambas configuraciones contra el ground truth. **Requiere el modelo VisDrone.**
Usamos un subconjunto (`--limit`) para que corra rápido en vivo.

In [ ]:
# Solo tiene sentido con el modelo entrenado en VisDrone
!python src/evaluate.py --model $MODEL --limit 80

import pandas as pd
pd.read_csv('outputs/benchmark_map.csv')

## 6. Experimento en vivo: tamaño del slice

Slices más pequeños = detectas objetos más diminutos, pero más lento. El trade-off clásico.

In [ ]:
import time, glob
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

m = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=MODEL,
                                       confidence_threshold=0.25, device='cuda:0')
img = sorted(glob.glob('data/VisDrone2019-DET-val/images/*.jpg'))[0]
for s in [320, 512, 640, 768]:
    t0 = time.perf_counter()
    r = get_sliced_prediction(img, m, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type='GREEDYNMM', verbose=0)
    print(f'slice={s}px -> {len(r.object_prediction_list):4d} detecciones en {time.perf_counter()-t0:.2f}s')

---
## Conclusiones

- YOLO + SAHI **sube el mAP** y, sobre todo, el **AP de objetos pequeños** — el caso de cámaras urbanas / vista aérea.
- Costo: **mayor tiempo de inferencia** (trade-off precisión vs velocidad).
- Todo **open source**: Ultralytics YOLO + SAHI + pycocotools, dataset VisDrone abierto.